## Aeropulse — Orchestration: 04 Fail Batch

**Purpose:** Marks a batch `FAILED` and records the error, so it's visible in `batch_control` and can be retried or alerted on. Wire this to the *failure* path of any transform activity in the pipeline.

**Parameters (pipeline-injected):** `source_name`, `batch_id`, `error_message` (pass `@activity('<step>').Error.message` from the pipeline)


In [ ]:
# Parameters
# Injected by the pipeline's Notebook activity base parameters — defaults below are for standalone testing only.
source_name = "flight"
batch_id = ""
error_message = ""


In [ ]:
# 04 - fail batch
# escape single quotes so the error text can't break the SQL string, and cap the length so an
# unusually long stack trace doesn't blow past a reasonable column size
escaped_error = error_message.replace("'", "''")[:2000]

# flip the batch to FAILED, record when and why, and bump retry_count for visibility on repeated failures
spark.sql(f"""
    UPDATE control.batch_control
    SET status = 'FAILED', end_time = current_timestamp(),
        error_message = '{escaped_error}', retry_count = retry_count + 1,
        updated_timestamp = current_timestamp()
    WHERE source_name = '{source_name}' AND batch_id = '{batch_id}'
""")
